# CS229 Simplified PS2 — Classification and Neural Networks

参考 CS229 logistic stability、imbalanced labels 与 simple-NN 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：**先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

## Goal

对应 Spring 2026 Lecture 3–8。先在 logistic regression 的语境里正式引入 binary
classifier 和 cross-entropy，再研究类别不平衡，最后实现一个两层神经网络。

## Setup

`label` 为 0 或 1。正类样本较少，这是题目故意设置的实验条件。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps2_classification/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


In [ ]:
def load_classification(filename):
    data = np.loadtxt(DATA_DIR / filename, delimiter=",", skiprows=1)
    X = data[:, :2]
    y = data[:, 2].astype(int)
    X = np.column_stack([np.ones(len(X)), X])
    return X, y

X_train, y_train = load_classification("train.csv")
X_valid, y_valid = load_classification("valid.csv")
print("train:", X_train.shape, y_train.shape, "positive rate:", y_train.mean())
print("valid:", X_valid.shape, y_valid.shape, "positive rate:", y_valid.mean())


## Problem 1 — Logistic regression from scratch

**课堂连接：**Lecture 3，binary classification 与 logistic regression。

$$p(y=1\mid x;\theta)=\sigma(\theta^\top x),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.$$

先推导平均 binary cross-entropy 对 $\theta$ 的梯度，再实现稳定 sigmoid、loss 和
gradient descent。这里第一次正式使用 classifier；不是 Lecture 1 的内容。

> **你的答案（请双击本格后填写）**
>
> TODO：推导 binary cross-entropy gradient，并写出每个量的 shape。

In [ ]:
def sigmoid(z):
    # START TODO
    raise NotImplementedError("Implement a numerically stable sigmoid")
    # END TODO

def binary_cross_entropy(y, probability, sample_weight=None):
    # START TODO
    raise NotImplementedError("Clip probabilities and return weighted mean loss")
    # END TODO

def fit_logistic_regression(X, y, sample_weight=None, learning_rate=0.1, steps=2000):
    theta = np.zeros(X.shape[1])
    history = []
    for step in range(steps):
        # START TODO
        raise NotImplementedError("Compute probabilities, gradient, and update")
        # END TODO
    return theta, np.array(history)


In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    theta_log, history = fit_logistic_regression(X_train, y_train)
    probability = sigmoid(X_valid @ theta_log)
    prediction = (probability >= 0.5).astype(int)
    print("theta:", theta_log)
    print("first probabilities:", probability[:5])
    assert theta_log.shape == (3,)
    assert np.all((probability >= 0) & (probability <= 1))
    assert history[-1] < history[0]
else:
    print("P1 checks are off.")


## Problem 2 — 为什么 accuracy 会骗人

**题型来源：**新版公开作业中的 imbalanced classification 实验。

实现 TP、TN、FP、FN、accuracy、balanced accuracy、precision、recall 和 F1。
比较两个模型：全部预测为 0 的 baseline，以及 Problem 1 的 logistic regression。

**需要回答：**哪个指标揭示了少数类完全没被识别？不要只抄定义，要引用实际数字。

In [ ]:
def classification_metrics(y_true, y_pred):
    # Return a dictionary containing counts and requested metrics.
    # START TODO
    raise NotImplementedError("Compute confusion counts and metrics")
    # END TODO


In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    all_zero = np.zeros_like(y_valid)
    print("all-zero baseline:", classification_metrics(y_valid, all_zero))
    print("logistic model  :", classification_metrics(y_valid, prediction))
else:
    print("P2 checks are off. Complete and run P1 first.")


> **你的答案（请双击本格后填写）**
>
> TODO：引用你算出的数字，解释为什么 stakeholder 不能只看 accuracy。

## Problem 3 — 给少数类更高权重

为正类样本设置更大的 `sample_weight`，重新训练 logistic regression。比较普通模型与
加权模型的 majority/minority accuracy 和 balanced accuracy。

**预期：**少数类 recall 通常提高，但多数类 accuracy 可能下降。这是 trade-off，
不是“所有指标一起提高”。

In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    # START TODO
    raise NotImplementedError("Create weights, refit, and compare metrics")
    # END TODO
else:
    print("P3 checks are off.")


## Problem 4 — 两层神经网络处理 XOR

**课堂连接：**Lecture 7–8。线性决策边界无法解决 XOR，两层网络可以。

网络结构固定为 `2 -> 4 -> 1`，隐藏层用 `tanh`，输出用 sigmoid。先在纸上标出
$W_1,b_1,W_2,b_2$ 的 shape，再实现 forward 和 backward。

In [ ]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([0., 1., 1., 0.])

def init_mlp(rng):
    return {
        "W1": rng.normal(scale=0.5, size=(2, 4)),
        "b1": np.zeros(4),
        "W2": rng.normal(scale=0.5, size=(4, 1)),
        "b2": np.zeros(1),
    }

def mlp_forward(X, params):
    # START TODO
    raise NotImplementedError("Return probability and a cache for backprop")
    # END TODO

def mlp_backward(X, y, params, cache):
    # START TODO
    raise NotImplementedError("Return gradients matching every parameter shape")
    # END TODO


In [ ]:
RUN_P4_CHECKS = False
if RUN_P4_CHECKS:
    params = init_mlp(rng)
    for step in range(5000):
        probability, cache = mlp_forward(X_xor, params)
        grads = mlp_backward(X_xor, y_xor, params, cache)
        for name in params:
            assert grads[name].shape == params[name].shape
            params[name] -= 0.1 * grads[name]
    probability, _ = mlp_forward(X_xor, params)
    print("XOR probabilities:", probability.ravel())
    assert np.mean((probability.ravel() >= 0.5) == y_xor) == 1.0
else:
    print("P4 checks are off.")


## Checks

- [ ] 我能解释 classifier 为什么从 Lecture 3 才出现。
- [ ] sigmoid 和 cross-entropy 在极端输入下不产生 NaN。
- [ ] 我报告了 minority-class 指标，而不只报告 accuracy。
- [ ] backprop 返回的每个梯度与对应参数 shape 相同。

## Next Steps

完成后进入 PS3，处理没有标签或带隐变量的数据。